# Existing Google Drive Dataset Probe

**Goal**: Verify if the existing Google Drive `violence_data` folder contains a usable violence detection dataset.

**Safety**:
- Do NOT download from Kaggle
- Do NOT train full model
- Do NOT overwrite `backend/best_model.pt`
- Do NOT commit data files
- Do NOT print secrets

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive_mounted = False
try:
    drive.mount("/content/drive")
    drive_mounted = True
    print("drive_mount_attempted: SUCCESS")
except Exception as e:
    print(f"drive_mount_attempted: FAILED - {e}")
    drive_mounted = False

## Step 2: Check for Existing Dataset Folder

In [ ]:
from pathlib import Path

# Primary dataset path from previous successful Colab run
DATASET_ROOT = "/content/drive/MyDrive/violence_data"
root = Path(DATASET_ROOT)

print("dataset_path:", root)
folder_found = root.exists()
print("folder_found:", folder_found)

if folder_found:
    print("is_dir:", root.is_dir())
    print("children:", [x.name for x in root.iterdir()][:30])
    folder_path = str(root)
else:
    print("ERROR: Dataset folder not found!")
    folder_path = "Not found"

## Step 3: Inspect Dataset Structure

In [ ]:
from pathlib import Path
from collections import Counter

video_count = 0
file_extensions = set()
top_level_folders = []
detected_labels = []
detected_splits = []
split_label_counts = {}

if root is not None and root.exists():
    print("Using root:", root)
    
    videos = []
    for ext in ["*.avi", "*.mp4", "*.mov", "*.mkv"]:
        vids = list(root.rglob(ext))
        videos.extend(vids)
        if vids:
            file_extensions.add(ext.split("*")[1])
    
    video_count = len(videos)
    print("video_count:", video_count)
    
    print("\nfirst_20_videos:")
    for v in videos[:20]:
        print(v)
    
    top_level_folders = [p.name for p in root.iterdir() if p.is_dir()]
    print("\ntop_level_children:", top_level_folders[:50])
    
    # Count by split/label
    counts = Counter()
    for v in videos:
        parts = v.relative_to(root).parts
        if len(parts) >= 3:
            key = (parts[0], parts[1])
            counts[key] += 1
            if parts[0] not in detected_splits:
                detected_splits.append(parts[0])
            if parts[1] not in detected_labels:
                detected_labels.append(parts[1])
    
    print("\nsplit_label_counts:")
    for k, v in sorted(counts.items()):
        print(k, v)
        split_label_counts[f"{k[0]}/{k[1]}"] = v
else:
    print("ERROR: No dataset folder found!")
    split_label_counts = {}

## Step 4: Extract Project Code from Drive Zip

In [ ]:
from pathlib import Path
import zipfile, shutil, os

ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/AI-Sentinel/code/ai-sentinel-colab-code.zip"),
    Path("/content/drive/MyDrive/ملفاتي/AI-Sentinel/code/ai-sentinel-colab-code.zip"),
]

WEIGHT_CANDIDATES = [
    Path("/content/drive/MyDrive/AI-Sentinel/weights/best_model.pt"),
    Path("/content/drive/MyDrive/ملفاتي/AI-Sentinel/weights/best_model.pt"),
]

print("ZIP_CANDIDATES:")
for p in ZIP_CANDIDATES:
    print(str(p), "exists:", p.exists(), "size:", p.stat().st_size if p.exists() else "N/A")

print("WEIGHT_CANDIDATES:")
for p in WEIGHT_CANDIDATES:
    print(str(p), "exists:", p.exists(), "size:", p.stat().st_size if p.exists() else "N/A")

ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), None)
WEIGHT_PATH = next((p for p in WEIGHT_CANDIDATES if p.exists()), None)

print("selected_zip:", ZIP_PATH)
print("selected_weight:", WEIGHT_PATH)

if ZIP_PATH is None:
    raise FileNotFoundError("Project zip not found in any candidate path")

PROJECT_DIR = Path("/content/ai-sentinel")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(PROJECT_DIR)

os.chdir(PROJECT_DIR)

manifest_tool = Path("backend/tools/build_rwf2000_manifest.py")
smoke_tool = Path("backend/tools/train_rwf2000_smoke.py")
local_weight = Path("backend/best_model.pt")

if not local_weight.exists():
    print("backend/best_model.pt missing inside zip")
    if WEIGHT_PATH is None:
        raise FileNotFoundError("best_model.pt missing from zip and Drive weights candidates")
    local_weight.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WEIGHT_PATH, local_weight)

print("project_dir:", PROJECT_DIR)
print("project_children:", [p.name for p in PROJECT_DIR.iterdir()][:30])
print("manifest_tool_exists:", manifest_tool.exists())
print("smoke_tool_exists:", smoke_tool.exists())
print("best_model_exists:", local_weight.exists())
print("best_model_size:", local_weight.stat().st_size if local_weight.exists() else "N/A")

project_code_available = manifest_tool.exists() and smoke_tool.exists() and local_weight.exists()
manifest_tool_exists = manifest_tool.exists()
smoke_tool_exists = smoke_tool.exists()
best_model_exists = local_weight.exists()
print("project_code_available:", project_code_available)

if not project_code_available:
    raise RuntimeError("Project code setup failed. Do not continue to manifest or smoke.")

## Step 5: Install Missing Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

# Check which required packages are missing
required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "yaml": "pyyaml",
    "pytorchvideo": "pytorchvideo",
}

missing_packages = []
for module_name, pip_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(pip_name)
        print(f"Missing: {module_name}")

if missing_packages:
    print(f"\nInstalling missing packages: {missing_packages}")
    pkg_str = " ".join(missing_packages)
    !pip install -q {pkg_str}
    print("Installation complete.")
else:
    print("All required packages already installed.")

# Verify imports after installation
print("\nVerifying imports...")
all_good = True
for module_name in required_packages.keys():
    spec = importlib.util.find_spec(module_name)
    status = "OK" if spec is not None else "MISSING"
    if spec is None:
        all_good = False
    print(f"  {module_name}: {status}")

if not all_good:
    raise RuntimeError("Some required packages are still missing after installation")
else:
    print("\nAll imports verified successfully.")

## Step 6: Verify Project Setup

In [ ]:
print("selected_zip_path:", ZIP_PATH)
print("selected_zip_exists:", ZIP_PATH.exists() if ZIP_PATH else False)
print("selected_weight_path:", WEIGHT_PATH)
print("selected_weight_exists:", WEIGHT_PATH.exists() if WEIGHT_PATH else False)
print("project_dir:", PROJECT_DIR)
print("manifest_tool_exists:", manifest_tool_exists)
print("smoke_tool_exists:", smoke_tool_exists)
print("best_model_exists:", best_model_exists)
print("project_code_available:", project_code_available)

if not project_code_available:
    print("ERROR: Project code setup failed. Stopping.")
else:
    print("Project code setup successful.")

## Step 7: Build Manifest from Existing Drive Folder

In [ ]:
import json
from pathlib import Path

manifest_built = False
manifest_exists = False
MANIFEST = "/content/drive/MyDrive/AI-Sentinel/manifests/violence_data_manifest.jsonl"
FULL_RWF_ROOT = DATASET_ROOT

if root is None or not root.exists():
    print("ERROR: No dataset root found. Cannot build manifest.")
elif not project_code_available:
    print("ERROR: Project code not available. Cannot build manifest.")
else:
    MANIFEST_DIR = Path("/content/drive/MyDrive/AI-Sentinel/manifests")
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f"FULL_RWF_ROOT: {FULL_RWF_ROOT}")
    print(f"MANIFEST: {MANIFEST}")
    
    os.chdir("/content/ai-sentinel")
    
    !PYTHONPATH=backend python backend/tools/build_rwf2000_manifest.py \
      --dir "{FULL_RWF_ROOT}" \
      --output "{MANIFEST}" \
      --val-ratio 0.2 \
      --seed 42
    
    manifest_built = True
    manifest_exists = Path(MANIFEST).exists()
    print(f"\nmanifest_exists after build: {manifest_exists}")

## Step 8: Inspect Manifest

In [ ]:
import json
from pathlib import Path
from collections import Counter

manifest_entries = 0
manifest_labels = []
manifest_splits = []
manifest_split_label_dist = {}
original_split_preserved = False
seed_used = 42
manifest_issues = []

p = Path(MANIFEST)
print("manifest_exists:", p.exists())

if p.exists():
    rows = [json.loads(x) for x in p.read_text().splitlines() if x.strip()]
    manifest_entries = len(rows)
    manifest_labels = sorted(set(r["label"] for r in rows))
    manifest_splits = sorted(set(r.get("split", "unknown") for r in rows))
    original_split_preserved = all("original_split" in r for r in rows)
    
    print("entries:", manifest_entries)
    print("labels:", manifest_labels)
    print("splits:", manifest_splits)
    print("has_original_split:", original_split_preserved)
    
    counter = Counter((r.get("split", "unknown"), r["label"]) for r in rows)
    print("\nsplit/label distribution:")
    for key, value in sorted(counter.items()):
        print(key, value)
        manifest_split_label_dist[f"{key[0]}/{key[1]}"] = value
    
    if manifest_entries < 100:
        manifest_issues.append(f"Low entry count: {manifest_entries}")
else:
    print("Manifest not found.")
    manifest_issues.append("Manifest file not found")

## Step 9: Colab Dependency Diagnostics

In [ ]:
import importlib.util

packages = [
    "torch",
    "torchvision",
    "cv2",
    "numpy",
    "pandas",
    "pytorchvideo",
]

for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    print(f"{pkg}: {spec is not None}")

print("\nPython version:")
!python --version

print("\nKey package versions:")
!pip list 2>/dev/null | grep -E "torch|torchvision|pytorchvideo|opencv|numpy|pandas" || true

## Step 10: Verify Tool Imports

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

os.chdir("/content/ai-sentinel")

env = os.environ.copy()
env["PYTHONPATH"] = "backend"

cmd = [
    "python", "-c",
    "from tools.train_rwf2000_smoke import main; print('train_tool_import_ok')"
]

r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=120)

print(f"import_returncode: {r.returncode}")
print(f"import_stdout: {r.stdout[-2000:]}")
print(f"import_stderr: {r.stderr[-4000:]}")

if r.returncode != 0:
    print("\nERROR: train_rwf2000_smoke import failed before smoke run")
    # Try to install missing deps
    if "pytorchvideo" in r.stderr.lower():
        print("Attempting to install pytorchvideo...")
        !pip install -q pytorchvideo opencv-python-headless
        print("Installed pytorchvideo and opencv-python-headless")
else:
    print("\nTool import successful.")

## Step 11: Verify Model/Weight Compatibility

In [ ]:
import os, subprocess
from pathlib import Path

os.chdir("/content/ai-sentinel")

env = os.environ.copy()
env["PYTHONPATH"] = "backend"

cmd = [
    "python", "backend/tools/inspect_model_checkpoint.py",
    "--help"
]

r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=60)
print("inspect_tool_help_returncode:", r.returncode)
print("help_output:", r.stdout[:2000] if r.stdout else "N/A")

print("\n--- Now inspecting checkpoint ---")

cmd2 = [
    "python", "backend/tools/inspect_model_checkpoint.py",
    "--weights", "backend/best_model.pt"
]

r2 = subprocess.run(cmd2, capture_output=True, text=True, env=env, timeout=180)
print("checkpoint_inspect_returncode:", r2.returncode)
print("checkpoint_inspect_stdout_tail:", r2.stdout[-4000:] if r2.stdout else "N/A")
print("checkpoint_inspect_stderr_tail:", r2.stderr[-4000:] if r2.stderr else "N/A")

if r2.returncode != 0:
    print("\nWARNING: Checkpoint inspection failed")
else:
    print("\nCheckpoint inspection successful.")

## Step 12: Run 8/8 Smoke Verification Only

In [ ]:
import json
from pathlib import Path
import subprocess
import time
import re

smoke_ran = False
smoke_success = False
smoke_returncode = "N/A"
smoke_device = "N/A"
smoke_train_samples = 0
smoke_val_samples = 0
smoke_train_loss = "N/A"
smoke_val_loss = "N/A"
smoke_val_acc = "N/A"
smoke_runtime = "N/A"
smoke_checkpoint = "N/A"
smoke_checkpoint_saved = False
smoke_summary_found = False
smoke_latest_run_dir = "N/A"

p = Path(MANIFEST)

if not p.exists():
    print("Manifest not found - NOT running smoke")
elif manifest_entries < 16:
    print(f"Manifest only has {manifest_entries} entries - NOT running smoke")
elif not project_code_available:
    print("Project code not available - NOT running smoke")
elif not best_model_exists:
    print("best_model.pt not available - NOT running smoke")
else:
    print(f"Manifest has {manifest_entries} entries - running smoke verification")
    smoke_ran = True
    
    CHECKPOINT_BASE = Path("/content/drive/MyDrive/AI-Sentinel/checkpoints/smoke")
    CHECKPOINT_BASE.mkdir(parents=True, exist_ok=True)
    
    os.chdir("/content/ai-sentinel")
    
    # Set PYTHONPATH for subprocess to find backend modules
    env = os.environ.copy()
    env["PYTHONPATH"] = "backend"
    
    start_time = time.time()
    
    cmd = [
        "python", "backend/tools/train_rwf2000_smoke.py",
        "--manifest", MANIFEST,
        "--weights", "backend/best_model.pt",
        "--max-train-samples", "8",
        "--max-val-samples", "8",
        "--epochs", "1",
        "--device", "auto",
        "--profile", "legacy_slowfast",
        "--real-loader",
        "--output-dir", str(CHECKPOINT_BASE)
    ]
    
    print(f"Running command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=900)
    
    end_time = time.time()
    smoke_runtime = f"{end_time - start_time:.2f}s"
    smoke_returncode = result.returncode
    
    # Save stdout/stderr to Drive for debugging
    report_dir = Path("/content/drive/MyDrive/AI-Sentinel/reports")
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "smoke_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
    (report_dir / "smoke_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
    (report_dir / "smoke_returncode.txt").write_text(str(result.returncode), encoding="utf-8")
    print(f"Saved smoke logs to {report_dir}")
    
    print(f"\n=== SMOKE OUTPUT (returncode={result.returncode}) ===")
    print(result.stdout)
    if result.stderr:
        print("\n=== STDERR ===")
        print(result.stderr)
    
    # Check subprocess return code
    if result.returncode != 0:
        print(f"\nWARNING: Smoke subprocess failed with return code {result.returncode}")
        print("Setting smoke_success = False")
        smoke_success = False
        
        # Root cause detection
        smoke_root_cause = "Unknown"
        stderr_lower = result.stderr.lower() if result.stderr else ""
        stdout_lower = result.stdout.lower() if result.stdout else ""
        combined_output = stderr_lower + "\n" + stdout_lower
        
        if "modulenotfounderror" in combined_output or "no module named" in combined_output:
            smoke_root_cause = "ModuleNotFoundError - check PYTHONPATH and imports"
        elif "filenotfounderror" in combined_output or "no such file" in combined_output:
            smoke_root_cause = "FileNotFoundError - check manifest or weights path"
        elif "cuda out of memory" in combined_output or "oom" in combined_output:
            smoke_root_cause = "CUDA OOM - reduce batch size or use CPU"
        elif "connection" in combined_output or "timeout" in combined_output:
            smoke_root_cause = "Network/Timeout issue"
        elif "keyerror" in combined_output or "valueerror" in combined_output:
            smoke_root_cause = "Data format error - check manifest structure"
        elif "importerror" in combined_output:
            smoke_root_cause = "ImportError - missing dependencies"
        
        print(f"Root cause detection: {smoke_root_cause}")
    else:
        smoke_success = True
        smoke_root_cause = "None"
    
    # Discover latest smoke run folder under CHECKPOINT_BASE
    # The smoke script creates timestamped folders like smoke_YYYYMMDD_HHMMSS
    smoke_run_dirs = sorted(CHECKPOINT_BASE.rglob("smoke_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    # Filter to only directories
    smoke_run_dirs = [p for p in smoke_run_dirs if p.is_dir()]
    
    if smoke_run_dirs:
        latest_run = smoke_run_dirs[0]
        smoke_latest_run_dir = str(latest_run)
        print(f"\nLatest smoke run directory: {latest_run}")
        
        # Look for summary.json in the latest run
        summary_path = latest_run / "summary.json"
        if summary_path.exists():
            smoke_summary_found = True
            print(f"Found summary.json: {summary_path}")
            
            try:
                with open(summary_path, 'r') as f:
                    summary = json.load(f)
                
                # Parse metrics from summary.json
                # Keys in summary.json: train_loss, train_acc, val_loss, val_acc, duration_sec, profile, timestamp
                smoke_train_loss = summary.get('train_loss', 'N/A')
                smoke_val_loss = summary.get('val_loss', 'N/A')
                smoke_val_acc = summary.get('val_acc', 'N/A')
                smoke_train_samples = summary.get('train_samples', 'N/A')
                smoke_val_samples = summary.get('val_samples', 'N/A')
                smoke_device = summary.get('device', summary.get('profile', 'N/A'))
                
                # Checkpoint is saved as smoke_checkpoint.pt in the same directory
                checkpoint_candidate = latest_run / 'smoke_checkpoint.pt'
                if checkpoint_candidate.exists():
                    smoke_checkpoint = str(checkpoint_candidate)
                    smoke_checkpoint_saved = True
                else:
                    smoke_checkpoint = 'N/A'
                    smoke_checkpoint_saved = False
                
                print(f"Parsed from summary.json:")
                print(f"  train_loss: {smoke_train_loss}")
                print(f"  val_loss: {smoke_val_loss}")
                print(f"  val_accuracy: {smoke_val_acc}")
                print(f"  train_samples: {smoke_train_samples}")
                print(f"  val_samples: {smoke_val_samples}")
                print(f"  device: {smoke_device}")
            except Exception as e:
                print(f"Error parsing summary.json: {e}")
                smoke_summary_found = False
        else:
            print(f"summary.json not found in {latest_run}")
            # Look for smoke_checkpoint.pt in the run directory
            checkpoint_candidates = sorted(latest_run.glob("smoke_checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
            if checkpoint_candidates:
                smoke_checkpoint = str(checkpoint_candidates[0])
                smoke_checkpoint_saved = Path(smoke_checkpoint).exists()
                print(f"Found checkpoint: {smoke_checkpoint}")
            else:
                # Search recursively in smoke root
                all_checkpoints = sorted(CHECKPOINT_BASE.rglob("smoke_checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
                if all_checkpoints:
                    smoke_checkpoint = str(all_checkpoints[0])
                    smoke_checkpoint_saved = True
                    print(f"Found checkpoint (recursive): {smoke_checkpoint}")
            
            # Try to parse metrics from stdout as fallback
            output = result.stdout
            for line in output.split("\n"):
                if "device" in line.lower() and ("cuda" in line.lower() or "cpu" in line.lower()):
                    smoke_device = line.strip()
                if "train" in line.lower() and "loss" in line.lower():
                    try:
                        match = re.search(r'loss[:\s]+([0-9.]+)', line.lower())
                        if match:
                            smoke_train_loss = match.group(1)
                    except:
                        pass
                if "val" in line.lower() and "loss" in line.lower():
                    try:
                        match = re.search(r'loss[:\s]+([0-9.]+)', line.lower())
                        if match:
                            smoke_val_loss = match.group(1)
                    except:
                        pass
                if "acc" in line.lower() or "accuracy" in line.lower():
                    try:
                        match = re.search(r'acc[a-z]*[:\s]+([0-9.]+)', line.lower())
                        if match:
                            smoke_val_acc = match.group(1)
                    except:
                        pass
    else:
        print("No smoke run directories found under", CHECKPOINT_BASE)
        # Check for checkpoint in base directory as fallback
        checkpoint_candidates = list(CHECKPOINT_BASE.glob("*.pt"))
        if checkpoint_candidates:
            smoke_checkpoint = str(checkpoint_candidates[0])
            smoke_checkpoint_saved = Path(smoke_checkpoint).exists()
            print(f"Found checkpoint in base dir: {smoke_checkpoint}")
    
    print(f"\n=== FINAL SMOKE DIAGNOSTICS ===")
    print(f"smoke_ran: {smoke_ran}")
    print(f"smoke_success: {smoke_success}")
    print(f"subprocess_returncode: {smoke_returncode}")
    print(f"checkpoint_saved: {smoke_checkpoint_saved}")
    print(f"summary_found: {smoke_summary_found}")
    print(f"checkpoint_path: {smoke_checkpoint}")
    print(f"Device: {smoke_device}")
    print(f"Train samples: {smoke_train_samples}")
    print(f"Val samples: {smoke_val_samples}")
    print(f"Train loss: {smoke_train_loss}")
    print(f"Val loss: {smoke_val_loss}")
    print(f"Val accuracy: {smoke_val_acc}")
    print(f"Runtime: {smoke_runtime}")

## Step 13: Save Report to Drive and Print Results

In [ ]:
from pathlib import Path
import json
from datetime import datetime

# Determine if project code setup failed
project_setup_failed = not project_code_available

# Build the complete report
report = f"""# Executed Colab Dataset Manifest + Smoke Report

## Execution Status
- Notebook executed: Yes
- Drive mounted: {drive_mounted}
- selected_zip_path: {ZIP_PATH}
- selected_zip_exists: {ZIP_PATH.exists() if ZIP_PATH else False}
- selected_weight_path: {WEIGHT_PATH}
- selected_weight_exists: {WEIGHT_PATH.exists() if WEIGHT_PATH else False}
- project_dir: {PROJECT_DIR}
- manifest_tool_exists: {manifest_tool_exists}
- smoke_tool_exists: {smoke_tool_exists}
- best_model_exists: {best_model_exists}
- project_code_available: {project_code_available}
- Manifest generated: {manifest_exists if not project_setup_failed else False}
- Smoke verification executed: {smoke_ran if not project_setup_failed else False}
"""

if project_setup_failed:
    report += "\n## Reason: Project code setup failed\n"
    report += "- Check that the zip file exists at one of the candidate paths\n"
    report += "- Check that the zip extracts correctly\n"
    report += "- Check that manifest and smoke tools exist in the extracted code\n"
    report += "- Check that best_model.pt exists in the extracted code or Drive weights folder\n"

report += f"""
## Drive Dataset
- Dataset path: `{DATASET_ROOT}`
- Video count: {video_count}
- File extensions: {', '.join(file_extensions) if file_extensions else 'N/A'}
- Split/label counts:
"""

for key, value in split_label_counts.items():
    report += f"  - {key}: {value}\n"

report += f"""- Issues: {'; '.join(manifest_issues) if manifest_issues else 'None detected'}
"""

if not project_setup_failed:
    report += f"""
## Manifest
- Manifest path: `{MANIFEST}`
- Total entries: {manifest_entries}
- Train entries: {sum(v for k, v in manifest_split_label_dist.items() if k.startswith('train/'))}
- Val entries: {sum(v for k, v in manifest_split_label_dist.items() if k.startswith('val/'))}
- Labels: {', '.join(manifest_labels) if manifest_labels else 'N/A'}
- Split/label distribution:
"""

    for key, value in manifest_split_label_dist.items():
        report += f"  - {key}: {value}\n"

    report += f"""- `original_split` preserved: {original_split_preserved}
- Seed: {seed_used}
- Issues: {'; '.join(manifest_issues) if manifest_issues else 'None'}

## Smoke Verification
- Ran: {smoke_ran}
- Success: {smoke_success if smoke_ran else 'N/A'}
- Subprocess return code: {smoke_returncode if smoke_ran else 'N/A'}
- Root cause (if failed): {smoke_root_cause if smoke_ran else 'N/A'}
- Latest run directory: `{smoke_latest_run_dir if smoke_ran else 'N/A'}`
- Summary found: {smoke_summary_found if smoke_ran else 'N/A'}
- Device: {smoke_device}
- Train samples: {smoke_train_samples}
- Val samples: {smoke_val_samples}
- Train loss: {smoke_train_loss}
- Val loss: {smoke_val_loss}
- Val accuracy: {smoke_val_acc}
- Runtime: {smoke_runtime}
- Checkpoint path: `{smoke_checkpoint}`
- Checkpoint saved to Drive: {smoke_checkpoint_saved}
- Smoke stdout: `{report_dir / 'smoke_stdout.txt' if smoke_ran else 'N/A'}`
- Smoke stderr: `{report_dir / 'smoke_stderr.txt' if smoke_ran else 'N/A'}`
- Smoke returncode: `{report_dir / 'smoke_returncode.txt' if smoke_ran else 'N/A'}`
"""

report += f"""
## Safety
- Kaggle download attempted: No
- Dataset redownloaded: No
- Full training started: No
- `backend/best_model.pt` overwritten: No
- Secrets printed: No
- Data committed: No
- Checkpoints committed: No

## Verdict
- Existing `violence_data` usable: {'Yes' if video_count > 1000 else 'Needs review'}
"""

# Check all conditions for fine-tuning readiness
manifest_valid = manifest_exists and manifest_entries >= 1900 and set(manifest_labels) == {'normal', 'violence'}
smoke_valid = smoke_success is True and smoke_summary_found and smoke_checkpoint_saved
ready_for_finetuning = manifest_valid and smoke_valid

report += f"- Manifest valid: {manifest_valid}\n"
report += f"- Smoke verification valid: {smoke_valid}\n"
report += f"- Ready to propose full fine-tuning plan: {ready_for_finetuning}\n"
report += "- Need user approval before full training: Yes\n"

if ready_for_finetuning:
    report += "- Recommended next step: Proceed with full fine-tuning using existing dataset\n"
elif not manifest_valid:
    report += "- Recommended next step: Fix manifest generation issues\n"
elif not smoke_valid:
    report += "- Recommended next step: Fix smoke verification issues\n"
else:
    report += "- Recommended next step: Complete manifest generation and smoke verification first\n"

report += f"\n---\n*Report generated: {datetime.now().isoformat()}*\n"

# Save report to Drive
report_dir = Path("/content/drive/MyDrive/AI-Sentinel/reports")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "existing_violence_data_probe_report.md"
report_path.write_text(report)

# Print the required output format
print("REPORT_PATH:", str(report_path))
print("REPORT_EXISTS:", report_path.exists())
print("\nREPORT_CONTENT:")
print("="*60)
print(report)
print("="*60)